<a href="https://colab.research.google.com/github/felimaker/IBM-Data-Science-Capstone/blob/main/5_eda_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 5: EDA con SQL
**Proyecto:** Predicción de aterrizaje de la primera etapa del Falcon 9

**Objetivo:** Responder preguntas de negocio sobre el dataset de lanzamientos usando
consultas SQL, cargando los datos en una base **SQLite en memoria** (equivalente
ligero a Db2, sin necesidad de credenciales ni instalación).

Repositorio de GitHub: **https://github.com/felimaker/IBM-Data-Science-Capstone**


In [2]:
import pandas as pd
import sqlite3

# Descargamos los datos directamente desde la URL de respaldo
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
df = pd.read_csv(url)

conn = sqlite3.connect(':memory:')
df.to_sql('SPACEXTABLE', conn, index=False, if_exists='replace')

%load_ext sql
%sql sqlite:///:memory:


> **Nota:** en Colab, si `%sql` no reconoce la conexión en memoria, usa directamente
> `pd.read_sql_query(query, conn)` como se muestra en cada celda de abajo — es el
> método usado en todo este notebook para máxima compatibilidad.


## 1. Nombres únicos de los sitios de lanzamiento

In [3]:
query = "SELECT DISTINCT LaunchSite FROM SPACEXTABLE;"
pd.read_sql_query(query, conn)


,LaunchSite
0,CCAFS SLC 40
1,VAFB SLC 4E
2,KSC LC 39A


## 2. 5 registros donde el sitio de lanzamiento empiece con 'CCA'

In [4]:
query = """
SELECT * FROM SPACEXTABLE
WHERE LaunchSite LIKE 'CCA%'
LIMIT 5;
"""
pd.read_sql_query(query, conn)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0007,-80.577366,28.561857,0
3,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B1004,-80.577366,28.561857,0
4,6,2014-01-06,Falcon 9,3325.000000,GTO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B1005,-80.577366,28.561857,0


## 3. Masa total de carga útil transportada por clientes de la NASA (CRS)

In [5]:
query = """
SELECT SUM(PayloadMass) AS TotalPayloadMass
FROM SPACEXTABLE
WHERE Orbit = 'ISS';
"""
pd.read_sql_query(query, conn)


,TotalPayloadMass
0,68878.7


## 4. Masa promedio de carga útil transportada por la versión de cohete F9 v1.1

In [6]:
query = """
SELECT AVG(PayloadMass) AS AvgPayloadMass
FROM SPACEXTABLE
WHERE BoosterVersion = 'Falcon 9';
"""
pd.read_sql_query(query, conn)


,AvgPayloadMass
0,6104.959412


## 5. Fecha del primer aterrizaje exitoso en plataforma marítima (drone ship)

In [7]:
query = """
SELECT MIN(Date) AS FirstSuccessfulDroneshipLanding
FROM SPACEXTABLE
WHERE Outcome LIKE '%True ASDS%';
"""
pd.read_sql_query(query, conn)


,FirstSuccessfulDroneshipLanding
0,2016-04-08


## 6. Nombres de los boosters con aterrizaje exitoso en plataforma terrestre y masa entre 4000-6000 kg

In [8]:
query = """
SELECT Serial, PayloadMass
FROM SPACEXTABLE
WHERE Outcome LIKE '%True RTLS%'
  AND PayloadMass BETWEEN 4000 AND 6000;
"""
pd.read_sql_query(query, conn)


,Serial,PayloadMass
0,B1040,4990.0


## 7. Número total de lanzamientos exitosos vs. fallidos (clasificación por Class)

In [9]:
query = """
SELECT Class,
       COUNT(*) AS TotalLaunches,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM SPACEXTABLE), 1) AS Percentage
FROM SPACEXTABLE
GROUP BY Class;
"""
pd.read_sql_query(query, conn)


,Class,TotalLaunches,Percentage
0,0,30,33.3
1,1,60,66.7


## 8. Sitio de lanzamiento con mayor número de lanzamientos exitosos

In [10]:
query = """
SELECT LaunchSite, COUNT(*) AS SuccessfulLaunches
FROM SPACEXTABLE
WHERE Class = 1
GROUP BY LaunchSite
ORDER BY SuccessfulLaunches DESC;
"""
pd.read_sql_query(query, conn)


,LaunchSite,SuccessfulLaunches
0,CCAFS SLC 40,33
1,KSC LC 39A,17
2,VAFB SLC 4E,10


## 9. Clasificación de las versiones de booster que consiguieron mayor carga útil

In [11]:
query = """
SELECT Serial, PayloadMass
FROM SPACEXTABLE
WHERE PayloadMass = (SELECT MAX(PayloadMass) FROM SPACEXTABLE)
ORDER BY PayloadMass DESC;
"""
pd.read_sql_query(query, conn)


,Serial,PayloadMass
0,B1048,15600.0
1,B1051,15600.0
2,B1048,15600.0


## 10. Análisis temporal: aterrizajes fallidos en plataforma marítima durante 2015, por mes

In [12]:
query = """
SELECT strftime('%m', Date) AS Month, Outcome, LaunchSite
FROM SPACEXTABLE
WHERE strftime('%Y', Date) = '2015'
  AND Outcome LIKE '%False ASDS%';
"""
pd.read_sql_query(query, conn)


,Month,Outcome,LaunchSite
0,01,False ASDS,CCAFS SLC 40
1,04,False ASDS,CCAFS SLC 40


## 11. Ranking de los resultados de aterrizaje (Outcome) entre 2010-06-04 y 2017-03-20, ordenados por número de ocurrencias

In [13]:
query = """
SELECT Outcome, COUNT(*) AS Occurrences
FROM SPACEXTABLE
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Outcome
ORDER BY Occurrences DESC;
"""
pd.read_sql_query(query, conn)


,Outcome,Occurrences
0,None None,9
1,True ASDS,5
2,False ASDS,4
3,True RTLS,3
4,True Ocean,3
5,None ASDS,2
6,False Ocean,2


## Resumen de la EDA con SQL
Se ejecutaron **11 consultas SQL** sobre la tabla `SPACEXTABLE` cubriendo:
- **Sitios de lanzamiento**: nombres únicos y filtrado por prefijo (`LIKE 'CCA%'`).
- **Cargas útiles (payloads)**: suma total para misiones ISS, promedio para
  Falcon 9, y ranking de los boosters con mayor masa transportada.
- **Tasas de éxito**: conteo y porcentaje de lanzamientos exitosos vs. fallidos
  (`Class`), y sitio con más aterrizajes exitosos.
- **Clasificaciones (rankings)**: boosters con aterrizaje terrestre exitoso en un
  rango de masa específico, y ranking de resultados de aterrizaje por frecuencia.
- **Análisis temporal**: fecha del primer aterrizaje exitoso en dron marítimo, y
  aterrizajes fallidos en dron marítimo mes a mes durante 2015.

Repositorio de GitHub: **https://github.com/felimaker/IBM-Data-Science-Capstone**
